# BriefIt Clustering Evaluation Notebook
This notebook evaluates the DBSCAN clustering algorithm used in BriefIt on a hand-annotated dataset (`labeled_samples.json`).

In [13]:
import json
import itertools
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

import sys
import os
sys.path.append(os.path.abspath("..")) # Add root project to path to import ai_engine

from ai_engine.understanding.dedup_cluster import cluster_vectors, DBSCAN_EPS
from ai_engine.understanding.embedder import embed_batch

print("BriefIt Evaluation Dependencies Loaded")

BriefIt Evaluation Dependencies Loaded


## 1. Load the Evaluation Dataset
The `labeled_samples.json` file contains a representative set of news articles (titles and snippets). A human annotator has assigned a `true_cluster` ID to each article. Articles with the exact same `true_cluster` ID are covering the exact same real-world event (and thus should be merged into a single `Story` in the database). Different IDs indicate different events that should remain separate.

In [14]:
with open("labeled_samples.json") as f:
    labeled_samples = json.load(f)

texts = [f"{sample['title']}. {sample['snippet']}" for sample in labeled_samples]
y_true = [sample['true_cluster'] for sample in labeled_samples]

print(f"Total Articles Evaluated: {len(labeled_samples)}")

Total Articles Evaluated: 54


## 2. Generate Semantic Embeddings
We use the exact same `embed_batch` function from the `ai_engine` that the production pipeline uses. This leverages the `paraphrase-multilingual-MiniLM-L12-v2` model via `FastEmbed`.

In [15]:
print("Generating embeddings (this may take a moment)...")
embeddings = embed_batch(texts)
vectors = np.array(embeddings)
print(f"Generated vectors of shape {vectors.shape}")

Generating embeddings (this may take a moment)...
Generated vectors of shape (54, 384)


## 3. Run DBSCAN Clustering
We cluster the vectors using the configured `DBSCAN_EPS` (0.34) and cosine distance.

In [16]:
print(f"Running DBSCAN with eps={DBSCAN_EPS}...")
y_pred = cluster_vectors(vectors, eps=DBSCAN_EPS).tolist()
print(f"Found {len(set(y_pred))} clusters.")

Running DBSCAN with eps=0.34...
Found 37 clusters.


## 4. Compute Pairwise Metrics
We evaluate the clustering performance pairwise: for every possible pair of articles, did the algorithm correctly keep them together (True Positive) or correctly separate them (True Negative)?

In [17]:
n = len(y_true)
tp = fp = fn = 0
for i, j in itertools.combinations(range(n), 2):
    same_true = y_true[i] == y_true[j]
    same_pred = y_pred[i] == y_pred[j]
    
    if same_true and same_pred:
        tp += 1
    elif same_pred and not same_true:
        fp += 1
    elif same_true and not same_pred:
        fn += 1

precision = tp / (tp + fp) if (tp + fp) else 1.0
recall = tp / (tp + fn) if (tp + fn) else 1.0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) else 0.0

print(f"Pairwise True Positives (correctly merged): {tp}")
print(f"Pairwise False Positives (incorrectly merged): {fp} (Target is 0 to maintain trust)")
print(f"Pairwise False Negatives (incorrectly split): {fn}\n")

print(f"Clustering Precision: {precision:.3f}")
print(f"Clustering Recall:    {recall:.3f}")
print(f"Clustering F1 Score:  {f1:.3f}")

Pairwise True Positives (correctly merged): 26
Pairwise False Positives (incorrectly merged): 0 (Target is 0 to maintain trust)
Pairwise False Negatives (incorrectly split): 17

Clustering Precision: 1.000
Clustering Recall:    0.605
Clustering F1 Score:  0.754
